# OutputFixingParser

`OutputFixingParser`는 출력 파싱 중 형식 오류가 발생했을 때,
LLM을 다시 호출해서 잘못된 출력을 자동으로 고쳐주는 파서입니다.

예를 들어 `PydanticOutputParser`가 기대하는 JSON 형식이 아닌 결과가 나오면,
`OutputFixingParser`가 오류를 확인하고 올바른 형식으로 다시 수정하도록 요청합니다.

## 핵심 흐름

잘못된 출력
→ 파싱 실패
→ OutputFixingParser
→ LLM이 형식 수정
→ 정상적인 데이터로 파싱

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# LangSmith 추적 설정
from langchain_teddynote import logging

logging.langsmith("CH03-OutputParser")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List


class Actor(BaseModel):
    name: str = Field(description="name of an actor")
    film_names: List[str] = Field(
        description="list of names of films they starred in"
    )


actor_query = "Generate the filmography for a random actor."

# Actor 구조를 기준으로 출력 파서 생성
parser = PydanticOutputParser(pydantic_object=Actor)

In [ ]:
## 일부러 파싱 오류 만들기

아래 셀은 **오류가 발생하는 것이 정상**입니다.

JSON에서는 작은따옴표가 아니라 큰따옴표를 사용해야 하는데,
일부러 잘못된 형식의 데이터를 넣어봅니다.

In [ ]:
# 일부러 잘못된 형식 입력
misformatted = "{'name': 'Tom Hanks', 'film_names': ['Forrest Gump']}"

# 기존 PydanticOutputParser로 파싱 시도
parser.parse(misformatted)

In [ ]:
## OutputFixingParser로 오류 수정

이제 `OutputFixingParser`를 이용해서
잘못된 출력 형식을 LLM이 자동으로 수정하도록 합니다.

In [ ]:
from langchain_classic.output_parsers import OutputFixingParser

# 기존 parser를 감싸서 자동 수정 기능 추가
new_parser = OutputFixingParser.from_llm(
    parser=parser,
    llm=ChatOpenAI(model="gpt-4.1-mini")
)

In [ ]:
# 잘못된 원본 출력 확인
misformatted

In [ ]:
# OutputFixingParser로 잘못된 출력 수정 및 파싱
actor = new_parser.parse(misformatted)

In [ ]:
# 수정된 결과 확인
actor

In [ ]:
Actor(
    name='Tom Hanks',
    film_names=['Forrest Gump']
)